In [ ]:
project_path = "/home/jupyter"
import os
import sys

sys.path.append(project_path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
from google.cloud import bigquery

from fintrans_toolbox.src import bq_utils as bq
from fintrans_toolbox.src import table_utils as t

import ft_digital_trade.src.utils.read_data as read_utils
import ft_digital_trade.src.utils.clean_utils as clean_utils
import ft_digital_trade.src.utils.calculation_utils as calc_utils
import ft_digital_trade.src.utils.plot_utils as plot_utils

client = bigquery.Client()

In [ ]:
# Calculating Visa marketshare drop-off using change in cardholders over time
# Looks at how total number of UK cardholders in the dataset changes over time to scale each category of spend

cardholders = '''SELECT time_period_value, sum(cardholders) as total_cardholders
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel`
WHERE time_period = 'Quarter' 
  AND mcg = 'All' 
  AND mcc = 'All'
  AND merchant_channel = 'All'
  AND cardholder_origin = 'UNITED KINGDOM'
  AND cardholder_origin_country = 'All'
  GROUP BY time_period_value
  ORDER BY time_period_value ASC
'''
cardholders_total = bq.read_bq_table_sql(client, cardholders)
#cardholders_total

base_cardholders = cardholders_total['total_cardholders'].iloc[0]
#base_cardholder

cardholders_total['Change from Base'] = (base_cardholders / cardholders_total['total_cardholders'])
cardholders_total # Change from Base column can now be multiplied against each quarter's spend values to adjust the spend for Visa's marketshare

In [ ]:
# Getting a total online spend by mcg, by UK cardholders
online_by_mcg = '''SELECT time_period_value, sum(spend) as online_spend, mcg
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel`
WHERE time_period = 'Quarter'
  AND mcc = 'All'
  AND merchant_channel = 'Online'
  AND cardholder_origin = 'UNITED KINGDOM'
  AND cardholder_origin_country = 'All'
GROUP BY time_period_value, mcg
ORDER BY time_period_value  ASC'''
online_by_mcg_df = bq.read_bq_table_sql(client, online_by_mcg)
online_by_mcg_df

In [ ]:
# Getting a total f2f spend by mcg, by UK cardholders
f2f_by_mcg = '''SELECT time_period_value, sum(spend) as f2f_spend, mcg
FROM `ons-fintrans-data-prod.fintrans_visa.spend_origin_and_channel`
WHERE time_period = 'Quarter'
  AND mcc = 'All'
  AND merchant_channel = 'Face to Face'
  AND cardholder_origin = 'UNITED KINGDOM'
  AND cardholder_origin_country = 'All'
GROUP BY time_period_value, mcg
ORDER BY time_period_value  ASC'''
f2f_by_mcg_df = bq.read_bq_table_sql(client, f2f_by_mcg)
f2f_by_mcg_df

In [ ]:
# Adjusting the quarterly values for online spend values
merged_df = online_by_mcg_df.merge(cardholders_total, on='time_period_value', how='left')
merged_df['adjusted_spend'] = merged_df['online_spend'] * merged_df['Change from Base']

online_by_mcg_df['adjusted_online_spend'] = merged_df['adjusted_spend']
#online_by_mcg_df

In [ ]:
# Adjusting the quarterly values for f2f spend values
merged_df_f2f = f2f_by_mcg_df.merge(cardholders_total, on='time_period_value', how='left')
merged_df_f2f['adjusted_spend'] = merged_df_f2f['f2f_spend'] * merged_df_f2f['Change from Base']

f2f_by_mcg_df['adjusted_f2f_spend'] = merged_df_f2f['adjusted_spend']
f2f_by_mcg_df

In [ ]:
# Summing everything up on a yearly basis and formatting the quarterly data - for online values
quarterly_totals = online_by_mcg_df.sort_values(by =["mcg", "time_period_value"])

#quarterly_totals

In [ ]:
# Summing everything up on a yearly basis and formatting the quarterly data - for f2f values
quarterly_totals_f2f = f2f_by_mcg_df.sort_values(by =["mcg", "time_period_value"])

quarterly_totals_f2f

In [ ]:
# Calculating change values for online spending
quarterly_totals['nominal_change'] = quarterly_totals.groupby('mcg')['adjusted_online_spend'].diff()
quarterly_totals['percent_change'] = quarterly_totals.groupby('mcg')['adjusted_online_spend'].pct_change() * 100

quarterly_totals

In [ ]:
# Calculating change values for f2f spending
quarterly_totals_f2f['nominal_change'] = quarterly_totals_f2f.groupby('mcg')['adjusted_f2f_spend'].diff()
quarterly_totals_f2f['percent_change'] = quarterly_totals_f2f.groupby('mcg')['adjusted_f2f_spend'].pct_change() * 100

quarterly_totals_f2f

In [ ]:
all_nominal = quarterly_totals[quarterly_totals['mcg'] == 'All'][['time_period_value', 'nominal_change']].rename(columns={'nominal_change': 'all_nominal_change'})
all_nominal

In [ ]:
all_nominal_f2f = quarterly_totals_f2f[quarterly_totals_f2f['mcg'] == 'All'][['time_period_value', 'nominal_change']].rename(columns={'nominal_change': 'all_nominal_change'})
all_nominal_f2f

In [ ]:
# Merge to get 'All' nominal change for each year back in table
quarterly_totals = quarterly_totals.merge(all_nominal, on='time_period_value', how='left')

# Calculate contribution to 'All' nominal change
quarterly_totals['contribution_to_all_change'] = (quarterly_totals['nominal_change'] / quarterly_totals['all_nominal_change']) * 100
quarterly_totals

In [ ]:
# Merge to get 'All' nominal change for each year back in table
quarterly_totals_f2f = quarterly_totals_f2f.merge(all_nominal_f2f, on='time_period_value', how='left')

# Calculate contribution to 'All' nominal change
quarterly_totals_f2f['contribution_to_all_change'] = (quarterly_totals_f2f['nominal_change'] / quarterly_totals_f2f['all_nominal_change']) * 100
quarterly_totals_f2f

In [ ]:
quarterly_totals.to_csv('mcg_totals_quarterly.csv')
quarterly_totals_f2f.to_csv('f2f_mcg_totals_quarterly.csv')

In [ ]:
# Summing everything up on a yearly basis and formatting the quarterly data
drivers_for_online_spend = quarterly_totals #[["time_period_value", "mcg", "contribution_to_all_change","all_nominal_change","nominal_change"]]
drivers_for_online_spend = drivers_for_online_spend.sort_values(by = ["time_period_value", "contribution_to_all_change"])
drivers_for_online_spend

# Summing everything up on a yearly basis and formatting the quarterly data
drivers_for_f2f_spend = quarterly_totals_f2f #[["time_period_value", "mcg", "contribution_to_all_change","all_nominal_change","nominal_change"]]
drivers_for_f2f_spend = drivers_for_f2f_spend.sort_values(by = ["time_period_value", "contribution_to_all_change"])
drivers_for_f2f_spend

In [ ]:
drivers_for_f2f_spend.to_csv('f2f_drivers.csv')
drivers_for_online_spend.to_csv('online_drivers.csv')

In [ ]:
mcg_of_interest = ["DEPARTMENT STORES", "DISCOUNT STORES", "APPAREL & ACCESSORIES", "TRAVEL SERVICES", "EDUCATION & GOVERNMENT", "Airlines"]
filtered_df = drivers_for_online_spend[drivers_for_online_spend['mcg'].isin(mcg_of_interest)]
filtered_df.to_csv('selected_online_drivers.csv')

In [ ]:
f2f_mcg_of_interest = ["DEPARTMENT STORES", "DISCOUNT STORES", "APPAREL & ACCESSORIES", "RESTAURANTS", "RETAIL GOODS", "FOOD & GROCERY"]
filtered_df_f2f = drivers_for_f2f_spend[drivers_for_f2f_spend['mcg'].isin(f2f_mcg_of_interest)]
filtered_df_f2f.to_csv('selected_f2f_drivers.csv')